In [0]:
WITH notebook_usage AS (
  SELECT
    usage_date,
    usage_metadata.notebook_id AS notebook_id,
    usage_metadata.notebook_path AS notebook_path,
    SUM(usage_quantity) AS total_usage,
    identity_metadata.run_as AS user
  FROM
    system.billing.usage
  WHERE
    sku_name IN ('PREMIUM_ALL_PURPOSE_SERVERLESS_COMPUTE_US_EAST_2')
    AND usage_metadata.job_id IS NULL
    AND usage_metadata.notebook_id IS NOT NULL
    AND usage_metadata.notebook_path == '/Shared/Datamart/ACCOUNT_TRANSACTION/Validation Queries/Account-Daily/acct_dim/Prod - Acct_dim Validation Query'
  GROUP BY
    usage_date, 
    usage_metadata.notebook_path,
    identity_metadata.run_as,
    usage_metadata.notebook_id
)
SELECT
  n.usage_date, 
  n.notebook_id,
  n.notebook_path,
  n.user,
  (n.total_usage / t.total_usage) * t.total_cost AS notebook_cost
FROM
  notebook_usage n CROSS JOIN prd.value_stream_monitoring.sku_usage_cost t
WHERE
  t.sku = 'PREMIUM_ALL_PURPOSE_SERVERLESS_COMPUTE_US_EAST_2'
ORDER BY
  usage_date DESC
limit 10